# GitLab Catalog Discovery

GitLab Catalog Discovery durchsucht eine GitLab-Gruppe regelmässig nach `catalog-info.yaml`-Dateien und übernimmt die gefundenen Entities automatisch in den Backstage Software Catalog.

Das Modul ergänzt den Backstage Catalog um den GitLab Entity Provider, welcher GitLab-Projekte durchsucht und deren Catalog-Dateien einliest.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-catalog-backend-module-gitlab

## Backend-Modul registrieren

Das Catalog-Modul wird im Backstage-Backend registriert, damit der konfigurierte GitLab Provider beim Start geladen wird.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
grep -q "plugin-catalog-backend-module-gitlab" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-catalog-backend-module-gitlab'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-catalog-backend-module-gitlab

## GitLab Personal Access Token erstellen (optional)

In GitLab:

* Profilbild öffnen
* Edit profile
* Access → Personal access tokens → Generate legacy token
* Namen setzen, beispielsweise `backstage-catalog`
* Ablaufdatum setzen
* Scope `read_api` auswählen
* Token erstellen und sofort kopieren
* Token in [env-platen.py](../../data/env-platen.py) eintragen

In [ ]:
%%bash
source ~/data/env-platen.py

curl --fail --silent --show-error --header "PRIVATE-TOKEN: ${GITLAB_TOKEN}" "https://gitlab.com/api/v4/groups/ch-mc-b/projects?include_subgroups=true&per_page=100" | python3 -m json.tool

## GitLab Discovery Provider konfigurieren

Der Provider durchsucht die angegebene GitLab-Gruppe alle 30 Minuten nach `catalog-info.yaml`-Dateien und ignoriert archivierte sowie geforkte Projekte.

Als Gruppe verwenden wir [https://gitlab.com/ch-mc-b/autoshop-ms/app](https://gitlab.com/ch-mc-b/autoshop-ms/app)

**Hinweis**: Bei einer selbst betriebenen GitLab-Instanz müssen `host` und bei Bedarf `apiBaseUrl` angepasst werden.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
cat > app-config.gitlab.yaml <<'EOF'

integrations:
  gitlab:
    - host: gitlab.com
      token: ${GITLAB_TOKEN}

catalog:
  providers:
    gitlab:
      production:
        host: gitlab.com
        group: ch-mc-b/autoshop-ms/app
        entityFilename: catalog-info.yaml
        projectPattern: '[\s\S]*'
        skipForkedRepos: false
        includeArchivedRepos: false
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
  import:
    entityFilename: catalog-info.yaml
    pullRequestBranchName: backstage-integration
  rules:
    - allow:
        - Component
        - System
        - API
        - Resource
        - Location
        - Template
EOF


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Gitlab"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn backstage-cli config:print --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.test.yaml --config ~/mybackstage/app-config.gitlab.yaml 

## Backstage starten


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Gitlab"
export BACKSTAGE_PORT="3001"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.test.yaml --config ~/mybackstage/app-config.gitlab.yaml 2>&1 | tee /tmp/backstage-gitlab.log